# CLT-Forge Tutorial — GPT-2 & TinyStories (low-memory, T4/L4-friendly)

Three compute-light experiments on a **single Colab GPU** (T4/L4 fine):

1. **Activation loading speed** — on-the-fly vs. generate-and-store.
2. **CLT faithfulness** — explained variance, normalized MSE, L0, dead features.
3. **Compression & quantization standards** — fp16/bf16/int8/int4/int2 x zstd/lz4:
   bytes-per-token, save/load throughput, and reconstruction error.

The upstream configs in `runners/training/gpt2/config.py` target an **8xH100 node**
(`expansion_factor=32`, 300k steps, `feature_sharding` via `torchrun`, ~1-2 TB of
activations). Everything below is re-scaled to a single GPU. All the knobs live in
one cell (**Config**) so you can scale up freely.

**Runtime:** Colab -> Runtime -> Change runtime type -> GPU (T4 is enough).

## 0 · Setup

**Get the repo onto Colab first** — either upload `CLT-Forge-master.zip` via
the Colab file browser (drag into the `/content` folder) and run the cell below, or
swap in a `git clone` if you're working from a fork/remote.

In [ ]:
# Upload CLT-Forge-master.zip to /content first, then run this.
!unzip -q -o /content/CLT-Forge-master.zip -d /content

# Alternative: clone instead of upload
# !git clone <your-fork-url> /content/CLT-Forge-master


**IMPORTANT — numpy version conflict, do not skip this section.**
`sae-lens` pulls in `transformer-lens`, which wants `numpy<2` on Python 3.12 — but
Colab's entire preinstalled stack (jax, cupy, opencv, ...) needs `numpy>=2`. Letting
pip honor `transformer-lens`'s constraint downgrades numpy and breaks everything
else on the image, eventually crashing with `numpy.dtype size changed ... Expected
96, got 88` the first time something numpy2-compiled gets imported.

Fix: capture Colab's native numpy version, let the install temporarily downgrade it
(you'll see a wall of `requires numpy>=2, but you have numpy 1.26.4` conflict
warnings — expected, ignore them), force numpy back with `--no-deps`, then hard
**restart the kernel** so everything reloads consistently. Run the next 5 cells in
order, in one pass. Do not re-run the install cell after restoring numpy.

In [ ]:
# --- 1/5: capture Colab's native numpy version before touching anything ---
import subprocess
_native_numpy = subprocess.run(
    ["python", "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Colab's native numpy:", _native_numpy)


In [ ]:
# --- 2/5: install deps (Colab already has torch) ---
!pip -q install "sae-lens>=5.9.1,<6.11.2" "pydantic>=2.11,<3" zstandard tqdm datasets matplotlib wandb
!pip -q install lz4 || echo "lz4 optional — skipping"


In [ ]:
# --- 3/5: restore numpy to Colab's native version ---
# --no-deps: swap the files back without re-running dependency resolution
# (which would just try to re-downgrade it to satisfy transformer-lens again).
!pip -q install --force-reinstall --no-deps "numpy=={_native_numpy}"


In [ ]:
# --- 4/5: restart the kernel so everything reloads against the restored numpy ---
# Intentionally kills the process; Colab auto-restarts it. You WILL see
# "Your session crashed for an unknown reason" — expected, not an error.
# After it reconnects, run the VERIFY cell next.
import os
os.kill(os.getpid(), 9)


In [ ]:
# --- 5/5: verify before going further ---
import numpy
print(f"numpy {numpy.__version__}  ({numpy.__file__})")
assert int(numpy.__version__.split(".")[0]) >= 2, (
    f"numpy {numpy.__version__} is still <2 — the restore didn't take effect in this "
    "kernel. Re-run ONLY the restart cell above, wait for reconnect, then re-run this."
)
print("OK — numpy matches Colab's native version, safe to continue.")


In [ ]:
# --- Point Python at the CLT-Forge checkout ---
import os, sys

REPO = os.environ.get("CLT_FORGE_REPO", "/content/CLT-Forge-master")
assert os.path.isdir(REPO), (
    f"Set REPO to your CLT-Forge folder (got {REPO}). Unzip/clone the repo first."
)
sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, REPO)
print("Using repo:", REPO)

import torch
print("CUDA available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1 · Config — all the knobs

`EXPANSION_FACTOR=4` and `STEPS=4000` below are the validated settings from this
notebook's own runs (a smaller expansion factor OOM'd a T4 via `W_dec`'s Adam
state; the schedule lengths are scaled proportionally to `STEPS` rather than fixed,
matching `runners/training/gpt2/config.py`'s convention — see comments inline).

In [ ]:
from clt_forge.config.clt_training_runner_config import CLTTrainingRunnerConfig

# ---- experiment size (tune these) ----
MODEL              = "gpt2"
DATASET            = "apollo-research/roneneldan-TinyStories-tokenizer-gpt2"  # pre-tokenized for GPT-2
CONTEXT_SIZE       = 16
D_IN               = 768          # gpt2 d_model
EXPANSION_FACTOR   = 4            # 32 in the paper; cross-layer decoders are O(N_layers^2 * d_latent),
                                   # so even 8 (d_latent=6144) OOMs a T4 via Adam state on W_dec (~5.5 GiB)
DTYPE              = "float32"    # fp32 is simplest/deterministic on T4; bf16 needs ~2x lr
TRAIN_BATCH_TOKENS = 1024
STEPS              = 4000         # free to raise further — doesn't cost GPU memory, only wall-clock time

# on-the-fly buffer geometry
STORE_BATCH_PROMPTS  = 32
N_BATCHES_IN_BUFFER  = 4          # must be even, >= 2

# generate-and-store geometry
GEN_SPLITS           = 4
GEN_BUFFERS_PER_SPLIT= 4
N_TRAIN_BATCH_PER_BUF= 2          # cached buffer = this * TRAIN_BATCH_TOKENS

# schedule lengths, scaled proportionally to STEPS (matches runners/training/gpt2/config.py)
# instead of fixed absolute numbers — otherwise warmup/decay finish early and the rest of a
# longer run just trains flat instead of following a properly-shaped LR/L0 curve.
LR_WARMUP_STEPS    = max(10, STEPS // 20)
LR_DECAY_STEPS     = max(10, STEPS // 20)
L0_WAITING_STEPS   = 0
L0_WARMUP_STEPS    = int(0.7 * STEPS) - L0_WAITING_STEPS - 1
DECAY_STABLE_STEPS = STEPS - L0_WARMUP_STEPS - LR_DECAY_STEPS

ACT_PATH   = os.path.join(REPO, "storage", "activations", MODEL)
CKPT_PATH  = os.path.join(REPO, "storage", "checkpoints", MODEL)
os.makedirs(ACT_PATH, exist_ok=True)

# fields shared by every mode
_COMMON = dict(
    device="cuda", dtype=DTYPE, seed=42,
    n_checkpoints=0, checkpoint_path=CKPT_PATH, logger_verbose=True,
    model_class_name="HookedTransformer", model_name=MODEL,
    dataset_path=DATASET, is_dataset_tokenized=True, split="train",
    context_size=CONTEXT_SIZE, d_in=D_IN, expansion_factor=EXPANSION_FACTOR,
    jumprelu_init_threshold=0.03, jumprelu_bandwidth=1.0,
    train_batch_size_tokens=TRAIN_BATCH_TOKENS, gradient_accumulation_steps=1,
    adam_beta1=0.9, adam_beta2=0.999, lr=4e-4,
    lr_warm_up_steps=LR_WARMUP_STEPS, lr_decay_steps=LR_DECAY_STEPS,
    final_lr_scale=0.0, decay_stable_steps=DECAY_STABLE_STEPS,
    l0_coefficient=2.0,
    dead_penalty_coef=1e-4,        # dead features accelerate once L0 sparsifies late in training;
                                    # this keeps pressure against features going permanently silent
    dead_feature_window=250,
    l0_warm_up_steps=L0_WARMUP_STEPS, l0_waiting_steps=L0_WAITING_STEPS,
    optimal_l0=10,                 # matches the production config — early-stop once sparsity is
                                    # healthy instead of always running the full step count
    n_batches_for_norm_estimate=4,
    log_to_wandb=False,            # avoids the wandb_id requirement; we capture metrics ourselves
    distributed_setup="None",      # single GPU: no torchrun / feature_sharding
)

def make_cfg(mode, **overrides):
    # mode in {'onfly', 'generate', 'cached'}.
    # The config validators require fresh vs cached store params to be mutually
    # exclusive, so we only pass the ones that belong to each mode.
    p = dict(_COMMON)
    if mode in ("onfly", "generate"):
        p.update(store_batch_size_prompts=STORE_BATCH_PROMPTS,
                 n_batches_in_buffer=N_BATCHES_IN_BUFFER)
        # cached_activations_path left unset (fresh / on-the-fly)
    elif mode == "cached":
        p.update(cached_activations_path=ACT_PATH,
                 n_train_batch_per_buffer=N_TRAIN_BATCH_PER_BUF)
    else:
        raise ValueError(mode)
    p.update(overrides)
    return CLTTrainingRunnerConfig(**p)

# sanity: a fresh config builds
_ = make_cfg("onfly", total_training_tokens=STEPS * TRAIN_BATCH_TOKENS)
print("config OK  |  d_latent =", _.d_latent)
print(f"schedule   |  lr_warmup={LR_WARMUP_STEPS} lr_decay={LR_DECAY_STEPS} "
      f"l0_warmup={L0_WARMUP_STEPS} decay_stable={DECAY_STABLE_STEPS}")


## 2 · Experiment 1 — activation loading speed

`ActivationsStore` switches behaviour on `cfg.cached_activations_path`:

- **On-the-fly** (`None`): every buffer runs a GPT-2 forward pass to produce
  `blocks.{i}.ln2.hook_normalized` (in) and `blocks.{i}.hook_mlp_out` (out).
- **Generate-and-store**: `generate_and_save_activations` writes splits to disk
  once; training then streams them back (background prefetch thread), no forward.

We measure the training iterator's throughput (tokens/s) in each mode, plus the
one-time cost of generation, and compute the break-even reuse count.

In [ ]:
import time, torch
from sae_lens.load_model import load_model
from clt_forge.training.activations_store import ActivationsStore

def bench_store(store, n_batches=80, warmup=5):
    it = iter(store)
    for _ in range(warmup):
        next(it)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0, tok = time.time(), 0
    for _ in range(n_batches):
        out = next(it)           # (act_in, act_out) for CLT training config
        tok += out[-2].shape[0]  # act_in rows == tokens in the batch
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.time() - t0
    return tok / dt, dt, tok

# load GPT-2 once; reused across Experiment 1 (and Experiment 3, if run in the same session)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("HookedTransformer", MODEL, device=device, model_from_pretrained_kwargs=None)
print("model loaded")


In [ ]:
# --- On-the-fly throughput (first run downloads TinyStories, ~0.8 GB) ---
cfg_fly = make_cfg("onfly", total_training_tokens=STEPS * TRAIN_BATCH_TOKENS)
store_fly = ActivationsStore(model, cfg_fly)
fly_tps, fly_dt, fly_tok = bench_store(store_fly)
print(f"On-the-fly : {fly_tps:,.0f} tok/s  ({fly_tok} tokens in {fly_dt:.2f}s)")


In [ ]:
# --- Generate & save activation splits to disk (timed) ---
buffer_size_gen = STORE_BATCH_PROMPTS * CONTEXT_SIZE * N_BATCHES_IN_BUFFER
gen_tokens = GEN_SPLITS * GEN_BUFFERS_PER_SPLIT * buffer_size_gen
print(f"Generating {gen_tokens:,} tokens -> {GEN_SPLITS} splits "
      f"({GEN_BUFFERS_PER_SPLIT} buffers x {buffer_size_gen} tok each)")

cfg_gen = make_cfg("generate")
store_gen = ActivationsStore(model, cfg_gen)

t0 = time.time()
store_gen.generate_and_save_activations(
    path=ACT_PATH,
    split_count=GEN_SPLITS,
    number_of_tokens=gen_tokens,
    use_compression=False,       # Experiment 3 covers the compressed formats
)
gen_time = time.time() - t0
gen_tps = gen_tokens / gen_time
print(f"Generation : {gen_tps:,.0f} tok/s written  ({gen_time:.2f}s total)")


In [ ]:
# --- Cached (disk) throughput ---
cfg_cached = make_cfg("cached")
store_cached = ActivationsStore(model, cfg_cached)
cached_tps, cached_dt, cached_tok = bench_store(store_cached)
print(f"Cached load: {cached_tps:,.0f} tok/s  ({cached_tok} tokens in {cached_dt:.2f}s)")


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = os.path.join(REPO, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- data ----
E = list(range(1, 26))
onfly_cost  = [e / fly_tps for e in E]
cached_cost = [gen_time / gen_tokens + e / cached_tps for e in E]
breakeven = (gen_time / gen_tokens) / (1/fly_tps - 1/cached_tps)

# ---- save CSVs ----
throughput_df = pd.DataFrame({
    "mode": ["on-the-fly", "generate", "read"],
    "tokens_per_sec": [fly_tps, gen_tps, cached_tps],
})
throughput_df.to_csv(os.path.join(RESULTS_DIR, "exp1_throughput.csv"), index=False)

amortized_df = pd.DataFrame({
    "epoch": E,
    "on_the_fly_us_per_token": [c * 1e6 for c in onfly_cost],
    "caching_us_per_token": [c * 1e6 for c in cached_cost],
})
amortized_df.to_csv(os.path.join(RESULTS_DIR, "exp1_amortized_cost.csv"), index=False)
print("saved:", os.path.join(RESULTS_DIR, "exp1_throughput.csv"))
print("saved:", os.path.join(RESULTS_DIR, "exp1_amortized_cost.csv"))

# ---- plot 1: throughput bar chart ----
fig1, ax1 = plt.subplots(figsize=(5.5, 4), dpi=200)
ax1.bar(["on-the-fly", "generate", "read"],
        [fly_tps, gen_tps, cached_tps],
        color=["#4C78A8", "#F58518", "#54A24B"])
ax1.set_ylabel("tokens / s")
ax1.grid(alpha=.3, axis="y")
for i, v in enumerate([fly_tps, gen_tps, cached_tps]):
    ax1.text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
fig1.savefig(os.path.join(RESULTS_DIR, "exp1_throughput.png"), bbox_inches="tight")
fig1.savefig(os.path.join(RESULTS_DIR, "exp1_throughput.pdf"), bbox_inches="tight")
plt.show()

# ---- plot 2: amortized cost line chart ----
fig2, ax2 = plt.subplots(figsize=(5.5, 4), dpi=200)
ax2.plot(E, amortized_df["on_the_fly_us_per_token"], "-o", label="on-the-fly")
ax2.plot(E, amortized_df["caching_us_per_token"],   "-o", label="caching")
ax2.axvline(breakeven, color="gray", linestyle="--", alpha=.6)
ax2.annotate(f"break-even ≈ {breakeven:.1f} epochs", (breakeven, 0),
             xytext=(5, 15), textcoords="offset points", fontsize=9, color="gray")
ax2.set_xlabel("epochs of reuse"); ax2.set_ylabel("amortized µs / token")
ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR, "exp1_amortized_cost.png"), bbox_inches="tight")
fig2.savefig(os.path.join(RESULTS_DIR, "exp1_amortized_cost.pdf"), bbox_inches="tight")
plt.show()

speedup = cached_tps / fly_tps
print(f"Cached read is {speedup:.2f}x the on-the-fly throughput.")
print(f"Break-even at ~{breakeven:.1f} epochs of reuse — caching wins past that point.")


## 3 · Experiment 2 — CLT faithfulness

We train a small CLT on-the-fly and capture the reconstruction panel the trainer
already computes in `_build_train_step_log_dict`: **explained variance** (mean +
per layer), **normalized MSE** (`1 - EV`), **L0** (active features/token), and
**dead features**. `_log_train_step` only builds that dict when `log_to_wandb=True`,
so we patch it to build + record it every `wandb_log_frequency` steps with wandb off.

**IMPORTANT:** `CLT._initialize_b_enc` (next cell) assumes `d_latent > 10,000` when
picking each feature's initial bias — true at paper scale, not at this notebook's
downscaled `EXPANSION_FACTOR`. Unpatched, it raises `IndexError` the moment a CLT is
built. Run the patch cell below once, before creating any `CLTTrainingRunner`.

In [ ]:
import clt_forge.clt as _clt_module

def _initialize_b_enc_patched(self, hidden_pre):
    # rate-clamped: caps the target activation rate at 5% instead of the paper's
    # 10_000/d_latent (which exceeds 1, and so is invalid, once d_latent < ~10k)
    rate = min(10_000. / self.d_latent, 0.05)
    with torch.no_grad():
        thresh = torch.exp(self.log_threshold).detach().cpu()
        B = hidden_pre.shape[0]
        bias_values = torch.zeros_like(self.b_enc).detach().cpu()
        for layer in range(self.N_layers):
            for feature in range(self.local_d_latent):
                feature_pre_acts = hidden_pre[:, layer, feature]
                sorted_acts, _ = torch.sort(feature_pre_acts, descending=True)
                target_idx = min(int(rate * B) + 1, B - 1)
                threshold_value = sorted_acts[target_idx]
                bias_values[layer, feature] = thresh[layer, feature] - threshold_value
        self.b_enc.data = bias_values.to(self.device)

_clt_module.CLT._initialize_b_enc = _initialize_b_enc_patched
print("patched CLT._initialize_b_enc (rate-clamped, sparse init preserved)")


In [ ]:
import clt_forge.training.clt_trainer as ct

history = []
_KEEP = ("metrics/explained_variance", "metrics/normalized_mse", "metrics/l0",
         "metrics/dead_features", "losses/overall_loss")

def _log_train_step_capture(self, loss_metrics):
    if self.n_training_steps % self.cfg.wandb_log_frequency == 0:
        d = self._build_train_step_log_dict(loss_metrics)
        row = {k: (float(d[k].item()) if hasattr(d[k], "item") else float(d[k])) for k in _KEEP if k in d}
        row["step"], row["tokens"] = self.n_training_steps, self.n_tokens
        history.append(row)

ct.CLTTrainer._log_train_step = _log_train_step_capture
print("patched _log_train_step to capture metrics")


In [ ]:
from clt_forge.clt_training_runner import CLTTrainingRunner

history.clear()
cfg_train = make_cfg("onfly", total_training_tokens=STEPS * TRAIN_BATCH_TOKENS, wandb_log_frequency=10)
runner = CLTTrainingRunner(cfg_train)   # single-GPU: rank/world_size default to 0/1
runner.run()
print(f"done — captured {len(history)} log points")


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = os.path.join(REPO, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

steps = [h["step"] for h in history]
def col(k): return [h.get(k, float("nan")) for h in history]

history_df = pd.DataFrame({
    "step": steps,
    "explained_variance": col("metrics/explained_variance"),
    "normalized_mse": col("metrics/normalized_mse"),
    "l0": col("metrics/l0"),
    "dead_features": col("metrics/dead_features"),
})
history_df.to_csv(os.path.join(RESULTS_DIR, "exp2_faithfulness.csv"), index=False)
print("saved:", os.path.join(RESULTS_DIR, "exp2_faithfulness.csv"))

# Explained variance
fig1, ax1 = plt.subplots(figsize=(5.5, 4), dpi=200)
ax1.plot(steps, col("metrics/explained_variance"), "-", color="#4C78A8")
ax1.set_xlabel("step"); ax1.set_ylabel("explained variance")
ax1.grid(alpha=.3)
plt.tight_layout()
fig1.savefig(os.path.join(RESULTS_DIR, "exp2_explained_variance.png"), bbox_inches="tight")
fig1.savefig(os.path.join(RESULTS_DIR, "exp2_explained_variance.pdf"), bbox_inches="tight")
plt.show()

# Normalized MSE
fig2, ax2 = plt.subplots(figsize=(5.5, 4), dpi=200)
ax2.plot(steps, col("metrics/normalized_mse"), "-", color="#E45756")
ax2.set_xlabel("step"); ax2.set_ylabel("normalized MSE")
ax2.grid(alpha=.3)
plt.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR, "exp2_normalized_mse.png"), bbox_inches="tight")
fig2.savefig(os.path.join(RESULTS_DIR, "exp2_normalized_mse.pdf"), bbox_inches="tight")
plt.show()

# L0
fig3, ax3 = plt.subplots(figsize=(5.5, 4), dpi=200)
ax3.plot(steps, col("metrics/l0"), "-", color="#54A24B")
ax3.set_xlabel("step"); ax3.set_ylabel("L0 (active features / token)")
ax3.grid(alpha=.3)
plt.tight_layout()
fig3.savefig(os.path.join(RESULTS_DIR, "exp2_l0.png"), bbox_inches="tight")
fig3.savefig(os.path.join(RESULTS_DIR, "exp2_l0.pdf"), bbox_inches="tight")
plt.show()

# Dead features
fig4, ax4 = plt.subplots(figsize=(5.5, 4), dpi=200)
ax4.plot(steps, col("metrics/dead_features"), "-", color="#B279A2")
ax4.set_xlabel("step"); ax4.set_ylabel("dead features")
ax4.grid(alpha=.3)
plt.tight_layout()
fig4.savefig(os.path.join(RESULTS_DIR, "exp2_dead_features.png"), bbox_inches="tight")
fig4.savefig(os.path.join(RESULTS_DIR, "exp2_dead_features.pdf"), bbox_inches="tight")
plt.show()

last = history[-1]
print(f"Final  EV={last['metrics/explained_variance']:.4f}  "
      f"nMSE={last['metrics/normalized_mse']:.4f}  "
      f"L0={last['metrics/l0']:.2f}  dead={last['metrics/dead_features']:.1f}")
print("Note: EV is the faithfulness axis; L0 is the sparsity axis. The paper's "
      "runs trade them off over 300k steps — this short run just shows the curves.")


## 4 · Experiment 3 — compression & quantization standards

`CompressedActivationsStore` natively supports quantization {none(fp16), int8,
int4, int2} x compression {none, zstd(level), lz4}. We add **bf16** by hand (it
isn't in `CompressionConfig`'s quantization options — numpy has no native bf16
dtype, so we reinterpret its bits as `uint16` and reuse the same
`compress_bytes`/`decompress_bytes` backend). One real activation buffer is saved
and reloaded under each config, measuring bytes/token, save/load throughput, and
**reconstruction error** — relative L2 error, `||original - dequantized|| /
||original|| x 100`. Compression (zstd/lz4) is lossless, so within any quantization
level it only changes size/speed, never error — the error comes entirely from the
quantization step.

In [ ]:
import torch
from sae_lens.load_model import load_model
from clt_forge.training.activations_store import ActivationsStore

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if "model" not in globals():
    model = load_model("HookedTransformer", MODEL, device=device, model_from_pretrained_kwargs=None)
    print("model loaded")
else:
    print("reusing model from Experiment 1")


In [ ]:
import tempfile, time, numpy as np, torch
from pathlib import Path
from clt_forge.training.compressed_activations_store import (
    CompressedActivationsStore, CompressionConfig)

# grab one real (unnormalized) activation buffer on CPU
store_ref = ActivationsStore(model, make_cfg("onfly"))
ref_in, ref_out = store_ref._fresh_activation_batches(
    return_tokens=False, mix_with_previous_buffer=False)
ref_in, ref_out = ref_in.contiguous(), ref_out.contiguous()
n_tok = ref_in.shape[0]
n_layers, d = ref_in.shape[1], ref_in.shape[2]
fp16_bytes_per_tok = n_layers * d * 2 * 2  # in+out, 2 bytes
print(f"buffer: {n_tok} tokens x {n_layers} layers x {d} d  "
      f"| fp16 baseline = {fp16_bytes_per_tok} B/token")

def recon_ev(orig, deq):
    o, q = orig.float(), deq.float()
    err = (o - q).pow(2).sum()
    var = (o - o.mean()).pow(2).sum()
    ev  = 1 - (err / (var + 1e-12))
    rel = (o - q).norm() / (o.norm() + 1e-12)
    return float(ev), float(rel)

GRID = [
    ("fp16+none",  "none", "none", 0),
    ("fp16+zstd3", "none", "zstd", 3),
    ("fp16+zstd9", "none", "zstd", 9),
    ("fp16+lz4",   "none", "lz4",  0),
    ("int8+none",  "int8", "none", 0),
    ("int8+zstd3", "int8", "zstd", 3),
    ("int8+zstd9", "int8", "zstd", 9),
    ("int8+lz4",   "int8", "lz4",  0),
    ("int4+zstd3", "int4", "zstd", 3),
    ("int2+zstd3", "int2", "zstd", 3),
]

rows = []
tmp = Path(tempfile.mkdtemp())
for name, quant, comp, lvl in GRID:
    try:
        cfg = CompressionConfig(quantization=quant, compression=comp,
                                compression_level=(lvl if comp == "zstd" else 3))
        cs = CompressedActivationsStore(cfg)
        fp = tmp / f"{name.replace('+','_')}.bin"

        t0 = time.time()
        size = cs.save_compressed(path=fp, act_in=ref_in, act_out=ref_out, tokens=None)
        save_t = time.time() - t0
        t0 = time.time()
        din, dout, _, _ = cs.load_compressed(fp)
        load_t = time.time() - t0

        ev_in,  rel_in  = recon_ev(ref_in,  din)
        ev_out, rel_out = recon_ev(ref_out, dout)
        rows.append(dict(
            name=name,
            bytes_per_tok=size / n_tok,
            ratio=fp16_bytes_per_tok / (size / n_tok),
            save_mb_s=(ref_in.numel()+ref_out.numel())*4/1e6/save_t,
            load_mb_s=(ref_in.numel()+ref_out.numel())*4/1e6/load_t,
            ev_out=ev_out, rel_out=rel_out, ev_in=ev_in,
        ))
    except Exception as e:
        print(f"skip {name}: {e}")

print(f"{len(rows)} configs done")


In [ ]:
# bf16: not in CompressionConfig's quantization Literal, so quantize/dequantize by hand
BF16_GRID = [
    ("bf16+none",  "none", 0),
    ("bf16+zstd3", "zstd", 3),
    ("bf16+zstd9", "zstd", 9),
    ("bf16+lz4",   "lz4",  0),
]

def _bf16_bytes(t):
    # numpy has no native bfloat16 dtype, so reinterpret the bits as uint16 instead
    return t.to(torch.bfloat16).contiguous().view(torch.uint16).cpu().numpy().tobytes()

def _bf16_from_bytes(b, shape):
    arr = np.frombuffer(b, dtype=np.uint16).copy()
    return torch.from_numpy(arr).view(torch.bfloat16).view(shape).float()

for name, comp, lvl in BF16_GRID:
    try:
        cfg = CompressionConfig(quantization="none", compression=comp,
                                compression_level=(lvl if comp == "zstd" else 3))
        cs = CompressedActivationsStore(cfg)
        fp_in  = tmp / f"{name.replace('+','_')}_in.bin"
        fp_out = tmp / f"{name.replace('+','_')}_out.bin"

        t0 = time.time()
        comp_in  = cs.compress_bytes(_bf16_bytes(ref_in))
        comp_out = cs.compress_bytes(_bf16_bytes(ref_out))
        fp_in.write_bytes(comp_in); fp_out.write_bytes(comp_out)
        save_t = time.time() - t0
        size = fp_in.stat().st_size + fp_out.stat().st_size

        t0 = time.time()
        dec_in  = _bf16_from_bytes(cs.decompress_bytes(fp_in.read_bytes()),  ref_in.shape)
        dec_out = _bf16_from_bytes(cs.decompress_bytes(fp_out.read_bytes()), ref_out.shape)
        load_t = time.time() - t0

        ev_in,  rel_in  = recon_ev(ref_in,  dec_in)
        ev_out, rel_out = recon_ev(ref_out, dec_out)
        rows.append(dict(
            name=name,
            bytes_per_tok=size / n_tok,
            ratio=fp16_bytes_per_tok / (size / n_tok),
            save_mb_s=(ref_in.numel()+ref_out.numel())*4/1e6/save_t,
            load_mb_s=(ref_in.numel()+ref_out.numel())*4/1e6/load_t,
            ev_out=ev_out, rel_out=rel_out, ev_in=ev_in,
        ))
    except Exception as e:
        print(f"skip {name}: {e}")

print(f"{len(rows)} configs total (incl. bf16)")


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = os.path.join(REPO, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

df = pd.DataFrame(rows).set_index("name")

# ---- save CSV (error_% = relative L2 error; easiest single fidelity metric to read) ----
df_show = df.assign(**{
    "B/tok": df.bytes_per_tok.round(1),
    "ratio_x": df.ratio.round(2),
    "save_MB/s": df.save_mb_s.round(0),
    "load_MB/s": df.load_mb_s.round(0),
    "error_%": (df.rel_out * 100).round(2),
})[["B/tok", "ratio_x", "save_MB/s", "load_MB/s", "error_%"]]
df_show.to_csv(os.path.join(RESULTS_DIR, "exp3_compression_grid.csv"))
print("saved:", os.path.join(RESULTS_DIR, "exp3_compression_grid.csv"))

names = df.index.tolist()
error_pct = df.rel_out * 100

# bytes / token
fig1, ax1 = plt.subplots(figsize=(6.5, 4), dpi=200)
ax1.bar(names, df.bytes_per_tok, color="#4C78A8")
ax1.set_ylabel("bytes / token")
ax1.tick_params(axis='x', rotation=45)
ax1.grid(alpha=.3, axis="y")
plt.tight_layout()
fig1.savefig(os.path.join(RESULTS_DIR, "exp3_bytes_per_token.png"), bbox_inches="tight")
fig1.savefig(os.path.join(RESULTS_DIR, "exp3_bytes_per_token.pdf"), bbox_inches="tight")
plt.show()

# reconstruction error %
fig2, ax2 = plt.subplots(figsize=(6.5, 4), dpi=200)
ax2.bar(names, error_pct, color="#E45756")
ax2.set_ylabel("reconstruction error (%)")
ax2.tick_params(axis='x', rotation=45)
ax2.grid(alpha=.3, axis="y")
plt.tight_layout()
fig2.savefig(os.path.join(RESULTS_DIR, "exp3_reconstruction_error.png"), bbox_inches="tight")
fig2.savefig(os.path.join(RESULTS_DIR, "exp3_reconstruction_error.pdf"), bbox_inches="tight")
plt.show()

# compression vs fidelity trade-off
fig3, ax3 = plt.subplots(figsize=(6.5, 4), dpi=200)
ax3.scatter(df.bytes_per_tok, error_pct, s=60)
for n, x, y in zip(names, df.bytes_per_tok, error_pct):
    ax3.annotate(n, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax3.set_xlabel("bytes / token"); ax3.set_ylabel("reconstruction error (%)")
ax3.grid(alpha=.3)
plt.tight_layout()
fig3.savefig(os.path.join(RESULTS_DIR, "exp3_compression_tradeoff.png"), bbox_inches="tight")
fig3.savefig(os.path.join(RESULTS_DIR, "exp3_compression_tradeoff.pdf"), bbox_inches="tight")
plt.show()

print("Read the scatter: down-left of a point is strictly better (smaller AND lower error).")
print("int8 is usually the sweet spot; int4/int2 trade fidelity for extreme size cuts.")
df_show


## 5 · Notes & scaling up

- **On-the-fly vs cached:** on-the-fly pays the GPT-2 forward every buffer but has
  zero disk I/O and zero storage; caching pays a one-time write and then reads
  fast. Experiment 1's break-even plot shows exactly when reuse makes caching
  worth it — for many-epoch CLT training it almost always does.
- **Faithfulness:** EV / nMSE / L0 / dead-features are exactly what the trainer
  logs to wandb in a full run. `optimal_l0=10` lets a run stop early once sparsity
  is healthy rather than always burning the full step budget.
- **Compression:** int8+zstd is the repo's own default. int4/int2 trade fidelity
  for extreme size cuts; bf16 is a genuine floating-point alternative to fp16/int8
  worth comparing directly, not just a theoretical option.
- **Scaling up:** raise `STEPS` freely (no memory cost — only wall-clock time), or
  `EXPANSION_FACTOR` (mind `W_dec`'s O(N_layers^2 * d_latent) memory cost), and
  switch `DTYPE="bfloat16"` (with ~2x `lr`) on a bigger GPU. For multi-GPU, use the
  `torchrun` launcher in `runners/training/gpt2/launch_train.py` with
  `distributed_setup="feature_sharding"`.